# RGC Axon Targeting Image and Data Analysis Workflow

This notebook analyzes confocal microscopy images and annotation data from **in vivo focal electroporation** experiments in zebrafish RGCs.

## Goal

Quantify where electroporated RGC axons enter the optic tectum along the dorsal-ventral axis.

Main measurement:

```text
normalized_DV_position = 0-100%
```

where:

```text
0%   = ventral-most optic tract fascicle
100% = dorsal-most optic tract fascicle
```

This allows comparison across groups such as:

```text
control
nev mutant
rescue
GFP-only control
```

## Libraries and purpose

| Library | Purpose |
|---|---|
| `pathlib` | file/folder paths |
| `numpy` | numeric calculations |
| `pandas` | annotation tables and CSV export |
| `matplotlib` | image and result plotting |
| `seaborn` | group comparison plots |
| `skimage.io` | load TIFF microscopy images |
| `skimage.exposure` | normalize image contrast |
| `scipy.stats` | ANOVA/Kruskal statistical tests |
| `statsmodels` | mixed-effects model accounting for embryo |

Install:

```bash
pip install numpy pandas matplotlib seaborn scikit-image scipy statsmodels
```

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from skimage import io, exposure
from scipy import stats
import statsmodels.formula.api as smf

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 40)

sns.set_context("notebook")
sns.set_style("whitegrid")

## Expected folder structure

```text
rgc_axon_targeting_project/
    data/
        confocal_stacks/
        projections/
        annotations/
            axon_entry_annotations.csv
    outputs/
        projections/
        annotated_images/
        plots/
        tables/
```

## Annotation CSV columns

| Column | Meaning |
|---|---|
| `embryo_id` | larva/embryo ID |
| `axon_id` | axon/arbor ID |
| `group` | experimental group |
| `image_file` | matching image filename |
| `ventral_y` | y-position of ventral-most fascicle |
| `dorsal_y` | y-position of dorsal-most fascicle |
| `axon_entry_y` | y-position where axon enters tectum |
| `include` | TRUE/FALSE inclusion flag |
| `notes` | optional notes |

In [ ]:
project_dir = Path("../rgc_axon_targeting_project")

data_dir = project_dir / "data"
stack_dir = data_dir / "confocal_stacks"
projection_dir = data_dir / "projections"
annotation_dir = data_dir / "annotations"

output_dir = project_dir / "outputs"
output_projection_dir = output_dir / "projections"
annotated_image_dir = output_dir / "annotated_images"
plot_dir = output_dir / "plots"
table_dir = output_dir / "tables"

for folder in [
    output_projection_dir,
    annotated_image_dir,
    plot_dir,
    table_dir
]:
    folder.mkdir(parents=True, exist_ok=True)

annotation_csv = annotation_dir / "axon_entry_annotations.csv"

print("Project:", project_dir)
print("Annotation CSV:", annotation_csv)

# Step 1 — Create maximum-intensity projections

This converts a confocal z-stack into a 2D projection.

Purpose:

```text
z-stack → maximum intensity projection → easier axon visualization
```

In [ ]:
def load_image_stack(path):
    return io.imread(Path(path))


def create_max_projection(stack):
    arr = np.asarray(stack)

    if arr.ndim == 2:
        return arr

    # Assumes stack format is z, y, x.
    # Modify if your stack is channel, z, y, x.
    return np.max(arr, axis=0)


def normalize_for_display(image):
    image = image.astype(float)
    p_low, p_high = np.percentile(image, [1, 99])

    return exposure.rescale_intensity(
        image,
        in_range=(p_low, p_high),
        out_range=(0, 1)
    )


def save_max_projection(stack_path):
    stack_path = Path(stack_path)
    stack = load_image_stack(stack_path)
    projection = create_max_projection(stack)

    out_path = output_projection_dir / f"{stack_path.stem}_max_projection.tif"

    io.imsave(
        out_path,
        projection.astype(stack.dtype),
        check_contrast=False
    )

    return out_path


# Uncomment when stacks are available:
# for stack_path in sorted(stack_dir.glob("*.tif")):
#     out = save_max_projection(stack_path)
#     print("Saved:", out)

# Step 2 — Preview projection images

This verifies image quality and orientation before analysis.

In [ ]:
projection_files = sorted(output_projection_dir.glob("*.tif"))

if len(projection_files) == 0:
    projection_files = sorted(projection_dir.glob("*.tif"))

print("Projection files found:", len(projection_files))

n_show = min(len(projection_files), 6)

if n_show > 0:
    fig, axes = plt.subplots(1, n_show, figsize=(4 * n_show, 4))

    if n_show == 1:
        axes = [axes]

    for ax, path in zip(axes, projection_files[:n_show]):
        img = io.imread(path)
        ax.imshow(normalize_for_display(img), cmap="gray")
        ax.set_title(path.name, fontsize=8)
        ax.axis("off")

    plt.tight_layout()
    plt.show()
else:
    print("No projection images found yet.")

# Step 3 — Load annotation table

Only include axons that satisfy the experimental criteria:

1. GFP-positive RGC in dorsonasal retina
2. axon terminates in expected posteroventral tectum
3. axon entry point is clear
4. image quality is acceptable

In [ ]:
annotations = pd.read_csv(annotation_csv)

print("Raw annotations:", annotations.shape)
display(annotations.head())

required_cols = [
    "embryo_id",
    "axon_id",
    "group",
    "image_file",
    "ventral_y",
    "dorsal_y",
    "axon_entry_y",
    "include"
]

missing = [col for col in required_cols if col not in annotations.columns]

if missing:
    raise ValueError(f"Missing required columns: {missing}")

annotations["include"] = (
    annotations["include"]
    .astype(str)
    .str.lower()
    .isin(["true", "1", "yes", "y"])
)

for col in ["ventral_y", "dorsal_y", "axon_entry_y"]:
    annotations[col] = pd.to_numeric(annotations[col], errors="coerce")

included = annotations[
    annotations["include"]
].dropna(
    subset=["ventral_y", "dorsal_y", "axon_entry_y"]
).copy()

print("Included rows:", included.shape)
display(included.head())

# Step 4 — Calculate normalized D-V entry position

Formula:

```text
normalized_DV_position = 100 * (axon_entry_y - ventral_y) / (dorsal_y - ventral_y)
```

Interpretation:

```text
0%   = ventral-most fascicle
50%  = middle
100% = dorsal-most fascicle
```

If your image axis is inverted, set:

```python
INVERT_DV_AXIS = True
```

In [ ]:
def calculate_normalized_dv_position(row, invert=False):
    ventral = row["ventral_y"]
    dorsal = row["dorsal_y"]
    entry = row["axon_entry_y"]

    denom = dorsal - ventral

    if denom == 0:
        return np.nan

    value = 100 * (entry - ventral) / denom

    if invert:
        value = 100 - value

    return value


INVERT_DV_AXIS = False

included["normalized_DV_position"] = included.apply(
    lambda row: calculate_normalized_dv_position(row, invert=INVERT_DV_AXIS),
    axis=1
)

included["out_of_range"] = (
    (included["normalized_DV_position"] < 0)
    | (included["normalized_DV_position"] > 100)
)

display(
    included[
        [
            "embryo_id",
            "axon_id",
            "group",
            "normalized_DV_position",
            "out_of_range"
        ]
    ].head()
)

display(included[included["out_of_range"]])

# Step 5 — Save cleaned measurement table

This is the main quantitative output.

In [ ]:
clean_measurement_path = table_dir / "axon_entry_measurements_clean.csv"

included.to_csv(clean_measurement_path, index=False)

print("Saved:", clean_measurement_path)

# Step 6 — Annotation overlay QC images

This overlays:

```text
ventral line
dorsal line
axon entry point
```

on the projection image.

Purpose:

```text
visually verify measurements
```

In [ ]:
def find_image_file(image_file):
    image_file = Path(str(image_file)).name
    stem = Path(image_file).stem

    for folder in [output_projection_dir, projection_dir, stack_dir]:
        matches = list(folder.glob(image_file))
        if matches:
            return matches[0]

        matches = list(folder.glob(f"{stem}*"))
        if matches:
            return matches[0]

    return None


def plot_annotation_overlay(row):
    image_path = find_image_file(row["image_file"])

    if image_path is None:
        print("Image not found:", row["image_file"])
        return None

    img = io.imread(image_path)
    img = create_max_projection(img)
    img_disp = normalize_for_display(img)

    fig, ax = plt.subplots(figsize=(7, 7))

    ax.imshow(img_disp, cmap="gray")

    ax.axhline(row["ventral_y"], color="cyan", linestyle="--", label="ventral 0%")
    ax.axhline(row["dorsal_y"], color="magenta", linestyle="--", label="dorsal 100%")

    ax.scatter(
        img.shape[1] / 2,
        row["axon_entry_y"],
        color="yellow",
        s=80,
        label="axon entry"
    )

    ax.set_title(
        f'{row["embryo_id"]} axon {row["axon_id"]}\n'
        f'{row["group"]}: {row["normalized_DV_position"]:.1f}%'
    )

    ax.legend(loc="upper right")
    ax.axis("off")

    out_path = annotated_image_dir / f'{row["embryo_id"]}_{row["axon_id"]}_overlay.tif'

    fig.savefig(out_path, dpi=300, bbox_inches="tight")
    plt.close(fig)

    return out_path


# Uncomment to generate overlays:
# overlay_paths = []
# for _, row in included.iterrows():
#     out = plot_annotation_overlay(row)
#     if out is not None:
#         overlay_paths.append(out)
# print("Saved overlays:", len(overlay_paths))

# Step 7 — Group-level visualization

Compare normalized axon entry position across groups.

In [ ]:
plt.figure(figsize=(8, 5))

sns.boxplot(
    data=included,
    x="group",
    y="normalized_DV_position"
)

sns.stripplot(
    data=included,
    x="group",
    y="normalized_DV_position",
    color="black",
    alpha=0.5
)

plt.axhline(0, linestyle="--", color="gray")
plt.axhline(100, linestyle="--", color="gray")

plt.ylabel("Normalized D-V entry position (%)")
plt.xlabel("Group")
plt.title("RGC Axon Entry Position by Group")
plt.tight_layout()

out_path = plot_dir / "axon_entry_position_by_group.tif"

plt.savefig(out_path, dpi=300, bbox_inches="tight")
plt.show()

print("Saved:", out_path)

# Step 8 — Summary statistics

Calculates mean, median, standard deviation, SEM, and sample size per group.

In [ ]:
group_summary = (
    included
    .groupby("group")
    .agg(
        n_axons=("axon_id", "count"),
        n_embryos=("embryo_id", "nunique"),
        mean_DV_position=("normalized_DV_position", "mean"),
        std_DV_position=("normalized_DV_position", "std"),
        median_DV_position=("normalized_DV_position", "median"),
        min_DV_position=("normalized_DV_position", "min"),
        max_DV_position=("normalized_DV_position", "max")
    )
    .reset_index()
)

group_summary["sem_DV_position"] = (
    group_summary["std_DV_position"]
    / np.sqrt(group_summary["n_axons"])
)

summary_path = table_dir / "axon_entry_group_summary.csv"
group_summary.to_csv(summary_path, index=False)

display(group_summary)
print("Saved:", summary_path)

# Step 9 — Statistical tests

This performs:

```text
one-way ANOVA
Kruskal-Wallis test
```

Use Kruskal-Wallis if you do not want to assume normality.

In [ ]:
groups = [
    group["normalized_DV_position"].dropna().values
    for _, group in included.groupby("group")
]

if len(groups) >= 2:
    anova_stat, anova_p = stats.f_oneway(*groups)
    kruskal_stat, kruskal_p = stats.kruskal(*groups)

    stats_table = pd.DataFrame([
        {
            "test": "one_way_ANOVA",
            "statistic": anova_stat,
            "p_value": anova_p
        },
        {
            "test": "Kruskal_Wallis",
            "statistic": kruskal_stat,
            "p_value": kruskal_p
        }
    ])

    stats_path = table_dir / "axon_entry_group_stat_tests.csv"
    stats_table.to_csv(stats_path, index=False)

    display(stats_table)
    print("Saved:", stats_path)
else:
    print("Need at least two groups.")

# Step 10 — Mixed-effects model by embryo

If multiple axons are measured from the same embryo, they are not fully independent.

This model estimates:

```text
normalized_DV_position ~ group + embryo_effect
```

In [ ]:
try:
    mixed_model = smf.mixedlm(
        "normalized_DV_position ~ C(group)",
        data=included,
        groups=included["embryo_id"]
    ).fit()

    print(mixed_model.summary())

    mixed_path = table_dir / "axon_entry_mixed_effects_model_summary.txt"

    with open(mixed_path, "w") as f:
        f.write(mixed_model.summary().as_text())

    print("Saved:", mixed_path)

except Exception as e:
    print("Mixed-effects model could not be fit:")
    print(e)

# Step 11 — Embryo-level aggregation

This averages axons within each embryo before comparing groups.

Purpose:

```text
avoid over-weighting embryos with many measured axons
```

In [ ]:
embryo_summary = (
    included
    .groupby(["embryo_id", "group"])
    .agg(
        mean_DV_position=("normalized_DV_position", "mean"),
        n_axons=("axon_id", "count")
    )
    .reset_index()
)

embryo_summary_path = table_dir / "embryo_level_axon_entry_summary.csv"
embryo_summary.to_csv(embryo_summary_path, index=False)

display(embryo_summary.head())

plt.figure(figsize=(8, 5))

sns.boxplot(
    data=embryo_summary,
    x="group",
    y="mean_DV_position"
)

sns.stripplot(
    data=embryo_summary,
    x="group",
    y="mean_DV_position",
    color="black",
    alpha=0.6
)

plt.ylabel("Embryo mean normalized D-V position (%)")
plt.xlabel("Group")
plt.title("Embryo-Level Axon Entry Position")
plt.tight_layout()

out_path = plot_dir / "embryo_level_axon_entry_position_by_group.tif"

plt.savefig(out_path, dpi=300, bbox_inches="tight")
plt.show()

print("Saved:", embryo_summary_path)
print("Saved:", out_path)

# Final interpretation

The key output is:

```text
normalized_DV_position
```

Example interpretation:

```text
control mean = 20%
nev mutant mean = 55%
```

This suggests mutant axons enter the tectum at a more dorsal position than controls.

If rescue axons shift back toward control values, that supports rescue of the axon pathfinding phenotype.

## Final deliverables

```text
maximum-intensity projection images
annotation overlay QC images
clean axon measurement CSV
group summary CSV
statistical test CSV
mixed-effects model summary
group comparison plots
embryo-level summary plots
```